In [7]:
import torch

disp = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(disp)

cpu


## 1

In [8]:
from pathlib import Path
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from numpy.lib.stride_tricks import sliding_window_view
from sklearn.preprocessing import StandardScaler
from diffusers import DDPMScheduler, DDIMScheduler
import einops

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ==============================================================================
# 1. DATA LAYER AGNOSTICO (Tripla: target x, forzante u, parametri statici c)
# ==============================================================================

TARGET_COLS = ["pollution", "dew", "temp", "press", "wnd_spd"]
LOG_COLS = ["pollution", "wnd_spd"]
WND_CATEGORIES = ["NE", "SE", "NW", "cv"]
WINDOW_SIZE = 24

def prepare_pollution_splits(
    csv_train_path: str,
    csv_test_path: str,
    window_size: int = 24,
    val_ratio: float = 0.15,
):
    df_tr_raw = pd.read_csv(csv_train_path)
    df_te_raw = pd.read_csv(csv_test_path)

    def extract_arrays(df: pd.DataFrame, start_hour_offset: int = 0, start_day_offset: int = 0):
        df = df.copy()
        if "date" in df.columns:
            dt = pd.to_datetime(df["date"])
            hours = dt.dt.hour.values
            months = dt.dt.month.values
        else:
            idx = np.arange(len(df))
            hours = (idx + start_hour_offset) % 24
            months = (((idx // 24) + start_day_offset) % 365) // 30 + 1

        # Target x: (N, C_x)
        x_df = df[TARGET_COLS].copy()
        for col in LOG_COLS:
            x_df[col] = np.log1p(np.clip(x_df[col].values, 0.0, None))
        x_arr = x_df.values.astype(np.float32)

        # Forzante u: (N, C_u) -> ora ciclica (2) + wnd_dir one-hot (4) = 6 canali
        h_angle = 2.0 * np.pi * hours / 24.0
        hour_cov = np.stack([np.sin(h_angle), np.cos(h_angle)], axis=1)
        wnd_oh = np.stack(
            [(df["wnd_dir"].values == cat).astype(np.float32) for cat in WND_CATEGORIES],
            axis=1,
        )
        u_arr = np.concatenate([hour_cov, wnd_oh], axis=1).astype(np.float32)

        # Parametri statici di scenario c: (N, D_c) -> regime stagionale (sin/cos mese)
        m_angle = 2.0 * np.pi * (months - 1) / 12.0
        c_arr = np.stack([np.sin(m_angle), np.cos(m_angle)], axis=1).astype(np.float32)
        return x_arr, u_arr, c_arr

    x_full, u_full, c_full = extract_arrays(df_tr_raw)
    x_test, u_test, c_test = extract_arrays(df_te_raw, start_hour_offset=0, start_day_offset=0)

    # Split cronologico Train / Val con gap = window_size per evitare overlap leakage
    split_idx = int(len(x_full) * (1.0 - val_ratio))
    x_train, u_train, c_train = x_full[:split_idx], u_full[:split_idx], c_full[:split_idx]
    x_val, u_val, c_val = (
        x_full[split_idx + window_size :],
        u_full[split_idx + window_size :],
        c_full[split_idx + window_size :],
    )

    # Fit StandardScaler SOLO sul train (mantiene la scala dei picchi senza clipping)
    scaler = StandardScaler()
    x_train = scaler.fit_transform(x_train).astype(np.float32)
    x_val = scaler.transform(x_val).astype(np.float32)
    x_test = scaler.transform(x_test).astype(np.float32)

    return (x_train, u_train, c_train), (x_val, u_val, c_val), (x_test, u_test, c_test), scaler


class ConditionedTrajectoryDataset(Dataset):
    """Dataset agnostico per finestre multivariate (x, u, c)."""

    def __init__(
        self,
        x_2d: np.ndarray,
        u_2d: np.ndarray,
        c_2d: np.ndarray,
        window_size: int = 24,
        stride: int = 1,
    ):
        # sliding_window_view su axis=0 produce shape (N_win, C, T)
        self.x = sliding_window_view(x_2d, window_shape=window_size, axis=0)[::stride].astype(np.float32)
        self.u = sliding_window_view(u_2d, window_shape=window_size, axis=0)[::stride].astype(np.float32)
        # Il vettore statico c della finestra viene preso al centro della finestra
        c_win = sliding_window_view(c_2d, window_shape=window_size, axis=0)[::stride]
        self.c = c_win[:, :, window_size // 2].astype(np.float32)  # (N_win, D_c)

    def __len__(self) -> int:
        return len(self.x)

    def __getitem__(self, idx: int) -> dict[str, torch.Tensor]:
        return {
            "x": torch.from_numpy(self.x[idx]),  # (C_x, T)
            "u": torch.from_numpy(self.u[idx]),  # (C_u, T)
            "c": torch.from_numpy(self.c[idx]),  # (D_c,)
        }

# ------------------------------------------------------------------------------
# Esplorazione dati
# ------------------------------------------------------------------------------

def inspect_dataset_summary(train_ds: ConditionedTrajectoryDataset, val_ds: ConditionedTrajectoryDataset, test_ds: ConditionedTrajectoryDataset):
    print(f"Finestre -> Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")
    sample = train_ds[0]
    print(f"Forme singolo campione -> x: {tuple(sample['x'].shape)}, u: {tuple(sample['u'].shape)}, c: {tuple(sample['c'].shape)}")
    print(f"Check NaN/Inf -> x: {np.isnan(train_ds.x).any()}, u: {np.isnan(train_ds.u).any()}, c: {np.isnan(train_ds.c).any()}")
    print(f"Range valori scalati x -> min: {train_ds.x.min():.2f}, max: {train_ds.x.max():.2f}, std: {train_ds.x.std():.2f}")

# ------------------------------------------------------------------------------

## 2

In [9]:
import torch
import torch.nn as nn


class SinusoidalTimeEmb(nn.Module):
    def __init__(self, dim: int):
        super().__init__()
        self.dim = dim

    def forward(self, timesteps: torch.Tensor) -> torch.Tensor:
        half = self.dim // 2
        freqs = torch.exp(
            -torch.log(torch.tensor(10000.0, device=timesteps.device))
            * torch.arange(half, device=timesteps.device)
            / half
        )
        args = timesteps[:, None].float() * freqs[None, :]
        return torch.cat([torch.sin(args), torch.cos(args)], dim=-1)


class FiLMResBlock1D(nn.Module):
    """Blocco residuo 1D modulato via FiLM (scale & shift) da timestep + parametri statici c."""

    def __init__(self, in_ch: int, out_ch: int, cond_dim: int, groups: int = 8):
        super().__init__()
        self.norm1 = nn.GroupNorm(groups, in_ch)
        self.act1 = nn.SiLU()
        self.conv1 = nn.Conv1d(in_ch, out_ch, kernel_size=3, padding=1, padding_mode="replicate")

        self.film_proj = nn.Sequential(
            nn.SiLU(),
            nn.Linear(cond_dim, out_ch * 2),
        )

        self.norm2 = nn.GroupNorm(groups, out_ch)
        self.act2 = nn.SiLU()
        self.conv2 = nn.Conv1d(out_ch, out_ch, kernel_size=3, padding=1, padding_mode="replicate")

        self.skip = nn.Conv1d(in_ch, out_ch, kernel_size=1) if in_ch != out_ch else nn.Identity()

    def forward(self, x: torch.Tensor, cond_emb: torch.Tensor) -> torch.Tensor:
        h = self.conv1(self.act1(self.norm1(x)))
        scale, shift = self.film_proj(cond_emb).chunk(2, dim=-1)
        h = self.norm2(h) * (1.0 + scale[:, :, None]) + shift[:, :, None]
        h = self.conv2(self.act2(h))
        return h + self.skip(x)


class TemporalSelfAttention1D(nn.Module):
    def __init__(self, channels: int, num_heads: int = 4, groups: int = 8):
        super().__init__()
        self.norm = nn.GroupNorm(groups, channels)
        self.attn = nn.MultiheadAttention(embed_dim=channels, num_heads=num_heads, batch_first=True)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.norm(x)
        h_seq = einops.rearrange(h, "b c t -> b t c")
        attn_out, _ = self.attn(h_seq, h_seq, h_seq)
        return x + einops.rearrange(attn_out, "b t c -> b c t")


class ConditionalUNet1D(nn.Module):
    """Backbone agnostico condizionato su maschera dinamica M, forzante u(t) e parametri statici c."""

    def __init__(
        self,
        x_channels: int,
        u_channels: int,
        c_dim: int,
        base_dim: int = 64,
        emb_dim: int = 128,
    ):
        super().__init__()
        self.time_mlp = nn.Sequential(
            SinusoidalTimeEmb(emb_dim),
            nn.Linear(emb_dim, emb_dim),
            nn.SiLU(),
            nn.Linear(emb_dim, emb_dim),
        )
        self.static_mlp = (
            nn.Sequential(
                nn.Linear(c_dim, emb_dim),
                nn.SiLU(),
                nn.Linear(emb_dim, emb_dim),
            )
            if c_dim > 0
            else None
        )

        # Canali in ingresso: x_noisy (C_x) + x_cond (C_x) + u (C_u) + mask (1)
        in_ch = x_channels * 2 + u_channels + 1
        self.in_conv = nn.Conv1d(in_ch, base_dim, kernel_size=3, padding=1, padding_mode="replicate")

        self.enc1 = FiLMResBlock1D(base_dim, base_dim, emb_dim)
        self.down1 = nn.Conv1d(base_dim, base_dim * 2, kernel_size=4, stride=2, padding=1)

        self.enc2 = FiLMResBlock1D(base_dim * 2, base_dim * 2, emb_dim)
        self.attn2 = TemporalSelfAttention1D(base_dim * 2, num_heads=4)
        self.down2 = nn.Conv1d(base_dim * 2, base_dim * 4, kernel_size=4, stride=2, padding=1)

        self.mid1 = FiLMResBlock1D(base_dim * 4, base_dim * 4, emb_dim)
        self.mid_attn = TemporalSelfAttention1D(base_dim * 4, num_heads=8)
        self.mid2 = FiLMResBlock1D(base_dim * 4, base_dim * 4, emb_dim)

        self.up2 = nn.ConvTranspose1d(base_dim * 4, base_dim * 2, kernel_size=4, stride=2, padding=1)
        self.dec2 = FiLMResBlock1D(base_dim * 4, base_dim * 2, emb_dim)
        self.dec_attn2 = TemporalSelfAttention1D(base_dim * 2, num_heads=4)

        self.up1 = nn.ConvTranspose1d(base_dim * 2, base_dim, kernel_size=4, stride=2, padding=1)
        self.dec1 = FiLMResBlock1D(base_dim * 2, base_dim, emb_dim)

        self.out_norm = nn.GroupNorm(8, base_dim)
        self.out_act = nn.SiLU()
        self.out_conv = nn.Conv1d(base_dim, x_channels, kernel_size=3, padding=1, padding_mode="replicate")

    def forward(
        self,
        x_t: torch.Tensor,
        x_cond: torch.Tensor,
        u: torch.Tensor,
        mask: torch.Tensor,
        c: torch.Tensor | None,
        timestep: torch.Tensor,
    ) -> torch.Tensor:
        cond_emb = self.time_mlp(timestep)
        if self.static_mlp is not None and c is not None:
            cond_emb = cond_emb + self.static_mlp(c)

        h = torch.cat([x_t, x_cond, u, mask], dim=1)
        h0 = self.in_conv(h)

        e1 = self.enc1(h0, cond_emb)
        e2 = self.attn2(self.enc2(self.down1(e1), cond_emb))

        m = self.mid1(self.down2(e2), cond_emb)
        m = self.mid2(self.mid_attn(m), cond_emb)

        d2 = self.up2(m)
        d2 = self.dec_attn2(self.dec2(torch.cat([d2, e2], dim=1), cond_emb))

        d1 = self.up1(d2)
        d1 = self.dec1(torch.cat([d1, e1], dim=1), cond_emb)

        return self.out_conv(self.out_act(self.out_norm(d1)))


class ConditionalDiffusionProcess(nn.Module):
    """Process Layer modulare: incapsula training DDPM e sampling DDIM senza clipping."""

    def __init__(
        self,
        backbone: nn.Module,
        x_channels: int,
        num_train_timesteps: int = 500,
    ):
        super().__init__()
        self.backbone = backbone
        self.x_channels = x_channels
        self.num_train_timesteps = num_train_timesteps

        self.train_scheduler = DDPMScheduler(
            num_train_timesteps=num_train_timesteps,
            beta_schedule="squaredcos_cap_v2",
            clip_sample=False,
        )
        self.infer_scheduler = DDIMScheduler(
            num_train_timesteps=num_train_timesteps,
            beta_schedule="squaredcos_cap_v2",
            clip_sample=False,
        )

    @staticmethod
    def sample_dynamic_mask(bsz: int, t_len: int, device: torch.device) -> torch.Tensor:
        """Alterna: (1) generazione pura da (c, u) [mask=0], (2) prefix forecasting, (3) inpainting."""
        mask = torch.zeros((bsz, 1, t_len), device=device)
        for b in range(bsz):
            r = random.random()
            if r < 0.34:
                # Modalità 1: Generazione completa condizionata solo su parametri statici c e forzante u
                pass
            elif r < 0.67:
                # Modalità 2: Forecasting da prefisso noto di lunghezza variabile [1 .. T//2]
                prefix_len = random.randint(1, max(1, t_len // 2))
                mask[b, 0, :prefix_len] = 1.0
            else:
                # Modalità 3: Inpainting (tutto noto tranne un buco centrale)
                mask[b, 0, :] = 1.0
                hole_len = random.randint(max(2, t_len // 4), max(3, t_len // 2))
                start = random.randint(0, t_len - hole_len)
                mask[b, 0, start : start + hole_len] = 0.0
        return mask

    def training_loss(
        self,
        x: torch.Tensor,
        u: torch.Tensor,
        c: torch.Tensor | None,
        mask: torch.Tensor | None = None,
    ) -> torch.Tensor:
        bsz, _, t_len = x.shape
        device = x.device

        if mask is None:
            mask = self.sample_dynamic_mask(bsz, t_len, device)

        timesteps = torch.randint(0, self.num_train_timesteps, (bsz,), device=device).long()
        noise = torch.randn_like(x)

        noisy_x = self.train_scheduler.add_noise(x, noise, timesteps)
        # Blocca le posizioni osservate al valore reale pulito
        noisy_x = noisy_x * (1.0 - mask) + x * mask
        x_cond = x * mask

        pred_noise = self.backbone(noisy_x, x_cond, u, mask, c, timesteps)

        target_mask = (1.0 - mask).expand_as(noise)
        sq_error = (pred_noise - noise) ** 2 * target_mask
        return sq_error.sum() / target_mask.sum().clamp(min=1.0)

    @torch.no_grad()
    def sample(
        self,
        u: torch.Tensor,
        c: torch.Tensor | None,
        x_known: torch.Tensor | None = None,
        mask: torch.Tensor | None = None,
        num_steps: int = 50,
    ) -> torch.Tensor:
        device = u.device
        bsz, _, t_len = u.shape

        if mask is None or x_known is None:
            mask = torch.zeros((bsz, 1, t_len), device=device, dtype=u.dtype)
            x_known = torch.zeros((bsz, self.x_channels, t_len), device=device, dtype=u.dtype)

        x_cond = x_known * mask
        x_t = torch.randn((bsz, self.x_channels, t_len), device=device, dtype=u.dtype)
        x_t = x_t * (1.0 - mask) + x_known * mask

        self.infer_scheduler.set_timesteps(num_steps, device=device)
        for t in self.infer_scheduler.timesteps:
            t_batch = torch.full((bsz,), t, device=device, dtype=torch.long)
            pred_noise = self.backbone(x_t, x_cond, u, mask, c, t_batch)
            x_t = self.infer_scheduler.step(pred_noise, t, x_t).prev_sample
            x_t = x_t * (1.0 - mask) + x_known * mask

        return x_t

## 3


In [10]:
def save_checkpoint(
    path: Path,
    process: ConditionalDiffusionProcess,
    optimizer: torch.optim.Optimizer,
    epoch: int,
    train_loss: float,
    val_loss: float,
    config: dict,
    scaler: StandardScaler,
):
    torch.save(
        {
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "model_state_dict": process.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "config": config,
            "scaler_mean_": scaler.mean_,
            "scaler_scale_": scaler.scale_,
            "scaler_var_": scaler.var_,
            "scaler_n_samples_seen_": scaler.n_samples_seen_,
        },
        path,
    )


@torch.no_grad()
def evaluate_loss(process: ConditionalDiffusionProcess, loader: DataLoader, device: torch.device) -> float:
    process.eval()
    total_loss, n_batches = 0.0, 0
    # Maschera fissa di validazione (es. generation pura mask=0) per confronto coerente tra epoche
    for batch in loader:
        x = batch["x"].to(device)
        u = batch["u"].to(device)
        c = batch["c"].to(device)
        val_mask = torch.zeros((x.shape[0], 1, x.shape[2]), device=device)
        loss = process.training_loss(x, u, c, mask=val_mask)
        total_loss += loss.item()
        n_batches += 1
    return total_loss / max(n_batches, 1)


def save_loss_curves(history: dict[str, list[float]], save_dir: Path):  # <-- AGGIUNTO
    """Salva lo storico numerico in CSV e il grafico delle loss in PNG su Drive."""
    epochs_range = range(1, len(history["train_loss"]) + 1)

    # 1. Salvataggio dati grezzi in CSV (utile per rifare i grafici in LaTeX per la tesi)
    df_hist = pd.DataFrame({
        "epoch": list(epochs_range),
        "train_loss": history["train_loss"],
        "val_loss": history["val_loss"],
    })
    df_hist.to_csv(save_dir / "loss_history.csv", index=False)

    # 2. Generazione e salvataggio grafico (scala lineare + scala logaritmica)
    fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

    for ax, scale in zip(axes, ["linear", "log"]):
        ax.plot(epochs_range, history["train_loss"], label="Train Loss", color="tab:blue", lw=2)
        ax.plot(epochs_range, history["val_loss"], label="Val Loss", color="tab:orange", lw=2)
        ax.set_yscale(scale)
        ax.set_title(f"Curva di Apprendimento (scala {scale})")
        ax.set_xlabel("Epoca")
        ax.set_ylabel("Masked MSE Loss")
        ax.grid(True, alpha=0.3)
        ax.legend()

    plt.tight_layout()
    fig.savefig(save_dir / "loss_curves.png", dpi=200)
    plt.savefig("drive/MyDrive/parameters/pollution_v2/diagnostics.png", dpi=200)
    plt.show()
    plt.close(fig)


def train_model(
    process: ConditionalDiffusionProcess,
    train_loader: DataLoader,
    val_loader: DataLoader,
    config: dict,
    scaler: StandardScaler,
    ckpt_dir: Path,
    epochs: int = 80,
    lr: float = 5e-4,
    device: torch.device = device,
):
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    best_path = ckpt_dir / "best_model.pt"

    process = process.to(device)
    optimizer = torch.optim.AdamW(process.parameters(), lr=lr, weight_decay=1e-4)
    scheduler_lr = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=lr * 0.02)

    best_val_loss = float("inf")
    history = {"train_loss": [], "val_loss": []}

    for epoch in range(1, epochs + 1):
        process.train()
        running_loss, n_batches = 0.0, 0

        for batch in train_loader:
            x = batch["x"].to(device)
            u = batch["u"].to(device)
            c = batch["c"].to(device)

            optimizer.zero_grad()
            loss = process.training_loss(x, u, c)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(process.parameters(), max_norm=1.0)
            optimizer.step()

            running_loss += loss.item()
            n_batches += 1

        scheduler_lr.step()
        train_loss = running_loss / max(num_batches if 'num_batches' in locals() else n_batches, 1)
        val_loss = evaluate_loss(process, val_loader, device)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            save_checkpoint(best_path, process, optimizer, epoch, train_loss, val_loss, config, scaler)
            tag = " <-- best"
        else:
            tag = ""

        if epoch % 5 == 0 or epoch == 1 or tag:
            print(f"Epoch {epoch:03d}/{epochs:03d} | Train: {train_loss:.5f} | Val: {val_loss:.5f}{tag}")

    save_loss_curves(history, ckpt_dir)
    return best_path

## 4

In [11]:
from scipy.stats import wasserstein_distance


def to_original_scale(traj_bct: torch.Tensor, scaler: StandardScaler) -> np.ndarray:
    arr_ntc = einops.rearrange(traj_bct.detach().cpu().numpy(), "n c t -> n t c")
    n_traj, t_steps, n_feat = arr_ntc.shape
    inv_ntc = scaler.inverse_transform(arr_ntc.reshape(-1, n_feat)).reshape(n_traj, t_steps, n_feat)
    for col in LOG_COLS:
        if col in TARGET_COLS:
            idx = TARGET_COLS.index(col)
            inv_ntc[:, :, idx] = np.expm1(np.clip(inv_ntc[:, :, idx], 0.0, 12.0))
    return inv_ntc


def evaluate_generative_fidelity(real_ntc: np.ndarray, gen_ntc: np.ndarray):
    # 1. Wasserstein 1D marginale
    w_dists = [
        wasserstein_distance(real_ntc[:, :, c].ravel(), gen_ntc[:, :, c].ravel())
        for c in range(real_ntc.shape[2])
    ]

    # 2. Errore PSD (Power Spectral Density) normalizzata lungo l'asse temporale
    real_psd = np.abs(np.fft.rfft(real_ntc, axis=1)) ** 2
    gen_psd = np.abs(np.fft.rfft(gen_ntc, axis=1)) ** 2
    real_psd /= real_psd.sum(axis=1, keepdims=True) + 1e-8
    gen_psd /= gen_psd.sum(axis=1, keepdims=True) + 1e-8
    psd_l1 = np.mean(np.abs(real_psd.mean(axis=0) - gen_psd.mean(axis=0)))

    # 3. Errore Cross-Correlazione tra canali (Frobenius norm)
    real_flat = real_ntc.reshape(-1, real_ntc.shape[2])
    gen_flat = gen_ntc.reshape(-1, gen_ntc.shape[2])
    corr_real = np.corrcoef(real_flat, rowvar=False)
    corr_gen = np.corrcoef(gen_flat, rowvar=False)
    corr_frob = np.linalg.norm(corr_real - corr_gen, ord="fro")

    print("\n=== VALIDAZIONE FEDELTÀ GENERATIVA (TEST SET UNIFORME) ===")
    for name, wd in zip(TARGET_COLS, w_dists):
        print(f"  Wasserstein [{name:10s}]: {wd:.4f}")
    print(f"  Errore Spettrale (PSD L1):  {psd_l1:.5f}")
    print(f"  Errore Cross-Corr (Frob):   {corr_frob:.5f}")


class InverseDiagnosticNet(nn.Module):
    """Rete diagnostica (Problema Inverso): stima i parametri statici nascosti c da (x, u)."""

    def __init__(self, in_channels: int, out_dim: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(in_channels, 64, kernel_size=3, padding=1),
            nn.SiLU(),
            nn.Conv1d(64, 128, kernel_size=3, stride=2, padding=1),
            nn.SiLU(),
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Linear(128, out_dim),
        )

    def forward(self, x: torch.Tensor, u: torch.Tensor) -> torch.Tensor:
        return self.net(torch.cat([x, u], dim=1))


def run_tstr_benchmark(
    x_train_syn: torch.Tensor,
    u_train_syn: torch.Tensor,
    c_train_syn: torch.Tensor,
    x_train_real_scarce: torch.Tensor,
    u_train_real_scarce: torch.Tensor,
    c_train_real_scarce: torch.Tensor,
    x_test_real: torch.Tensor,
    u_test_real: torch.Tensor,
    c_test_real: torch.Tensor,
    epochs: int = 25,
):
    def fit_and_eval(x_tr, u_tr, c_tr, label: str):
        model = InverseDiagnosticNet(x_tr.shape[1] + u_tr.shape[1], c_tr.shape[1]).to(device)
        opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
        ds = torch.utils.data.TensorDataset(x_tr, u_tr, c_tr)
        dl = DataLoader(ds, batch_size=64, shuffle=True)

        for _ in range(epochs):
            model.train()
            for bx, bu, bc in dl:
                bx, bu, bc = bx.to(device), bu.to(device), bc.to(device)
                loss = F.mse_loss(model(bx, bu), bc)
                opt.zero_grad()
                loss.backward()
                opt.step()

        model.eval()
        with torch.no_grad():
            pred_te = model(x_test_real.to(device), u_test_real.to(device))
            test_mse = F.mse_loss(pred_te, c_test_real.to(device)).item()
        print(f"  [{label:32s}] Test MSE stima parametri c: {test_mse:.5f}")

    print("\n=== BENCHMARK PROBLEMA INVERSO (TSTR & DATA AUGMENTATION) ===")
    fit_and_eval(x_train_real_scarce, u_train_real_scarce, c_train_real_scarce, "1. Solo Reali Scarsi (Baseline)")
    fit_and_eval(x_train_syn, u_train_syn, c_train_syn, "2. Solo Sintetici (TSTR)")
    fit_and_eval(
        torch.cat([x_train_real_scarce, x_train_syn], dim=0),
        torch.cat([u_train_real_scarce, u_train_syn], dim=0),
        torch.cat([c_train_real_scarce, c_train_syn], dim=0),
        "3. Reali Scarsi + Sintetici (Aug)",
    )


# ==============================================================================
# ESECUZIONE COMPLETA END-TO-END
# ==============================================================================
if __name__ == "__main__":
    train_tup, val_tup, test_tup, scaler = prepare_pollution_splits(
        "drive/MyDrive/datasets/pollution/multivariate_pollution.csv",
        "drive/MyDrive/datasets/pollution/pollution_test.csv",
        window_size=WINDOW_SIZE,
    )

    train_ds = ConditionedTrajectoryDataset(*train_tup, window_size=WINDOW_SIZE)
    val_ds = ConditionedTrajectoryDataset(*val_tup, window_size=WINDOW_SIZE)
    test_ds = ConditionedTrajectoryDataset(*test_tup, window_size=WINDOW_SIZE)

    train_loader = DataLoader(train_ds, batch_size=64, shuffle=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=128, shuffle=False)

    cfg = {
        "x_channels": len(TARGET_COLS),
        "u_channels": train_ds.u.shape[1],
        "c_dim": train_ds.c.shape[1],
        "base_dim": 64,
        "num_train_timesteps": 500,
        "window_size": WINDOW_SIZE,
    }

    backbone = ConditionalUNet1D(
        x_channels=cfg["x_channels"],
        u_channels=cfg["u_channels"],
        c_dim=cfg["c_dim"],
        base_dim=cfg["base_dim"],
    )
    process = ConditionalDiffusionProcess(
        backbone=backbone,
        x_channels=cfg["x_channels"],
        num_train_timesteps=cfg["num_train_timesteps"],
    )

    best_ckpt = train_model(
        process=process,
        train_loader=train_loader,
        val_loader=val_loader,
        config=cfg,
        scaler=scaler,
        ckpt_dir=Path("drive/MyDrive/parameters/pollution_v2"),
        epochs=80,
    )

    # Caricamento best model e valutazione uniforme su 500 scenari del test set
    ckpt = torch.load(best_ckpt, map_location=device, weights_only=False)
    process.load_state_dict(ckpt["model_state_dict"])
    process.eval()

    rng = np.random.default_rng(42)
    eval_idx = rng.choice(len(test_ds), size=min(500, len(test_ds)), replace=False)

    x_te_sub = torch.from_numpy(test_ds.x[eval_idx])
    u_te_sub = torch.from_numpy(test_ds.u[eval_idx])
    c_te_sub = torch.from_numpy(test_ds.c[eval_idx])

    # Generazione pura condizionata su (c, u) senza osservazioni di x (come per ENEA a t=0)
    gen_scaled_chunks = []
    for i in range(0, len(eval_idx), 128):
        gen_batch = process.sample(
            u=u_te_sub[i : i + 128].to(device),
            c=c_te_sub[i : i + 128].to(device),
            num_steps=50,
        )
        gen_scaled_chunks.append(gen_batch.cpu())
    gen_scaled = torch.cat(gen_scaled_chunks, dim=0)

    evaluate_generative_fidelity(
        real_ntc=to_original_scale(x_te_sub, scaler),
        gen_ntc=to_original_scale(gen_scaled, scaler),
    )

    # Benchmark TSTR: simuliamo scarsità di dati reali (es. solo 250 finestre dal train)
    scarce_idx = rng.choice(len(train_ds), size=250, replace=False)
    aug_idx = rng.choice(len(train_ds), size=2000, replace=False)
    u_aug = torch.from_numpy(train_ds.u[aug_idx])
    c_aug = torch.from_numpy(train_ds.c[aug_idx])

    syn_chunks = []
    for i in range(0, len(aug_idx), 128):
        syn_chunks.append(
            process.sample(u=u_aug[i : i + 128].to(device), c=c_aug[i : i + 128].to(device), num_steps=50).cpu()
        )
    x_syn = torch.cat(syn_chunks, dim=0)

    run_tstr_benchmark(
        x_train_syn=x_syn,
        u_train_syn=u_aug,
        c_train_syn=c_aug,
        x_train_real_scarce=torch.from_numpy(train_ds.x[scarce_idx]),
        u_train_real_scarce=torch.from_numpy(train_ds.u[scarce_idx]),
        c_train_real_scarce=torch.from_numpy(train_ds.c[scarce_idx]),
        x_test_real=torch.from_numpy(test_ds.x),
        u_test_real=torch.from_numpy(test_ds.u),
        c_test_real=torch.from_numpy(test_ds.c),
    )

KeyboardInterrupt: 

In [ ]:
def plot_thesis_diagnostics(
    process: ConditionalDiffusionProcess,
    test_ds: ConditionedTrajectoryDataset,
    scaler: StandardScaler,
    sample_idx: int = 10,
    n_ensemble: int = 15,
    vars_to_plot: list[str] = ["pollution", "temp", "press", "wnd_spd"],
):
    process.eval()
    u_single = torch.from_numpy(test_ds.u[sample_idx : sample_idx + 1]).to(device)
    c_single = torch.from_numpy(test_ds.c[sample_idx : sample_idx + 1]).to(device)
    x_real_bct = torch.from_numpy(test_ds.x[sample_idx : sample_idx + 1])

    # Genera K traiettorie diverse a partire dallo stesso scenario (c, u)
    u_rep = einops.repeat(u_single, "1 c t -> k c t", k=n_ensemble)
    c_rep = einops.repeat(c_single, "1 d -> k d", k=n_ensemble)
    gen_bct = process.sample(u=u_rep, c=c_rep, num_steps=50).cpu()

    real_ntc = to_original_scale(x_real_bct, scaler)[0]      # (T, C)
    gen_ntc = to_original_scale(gen_bct, scaler)            # (K, T, C)
    t_axis = np.arange(real_ntc.shape[0])

    indices = [TARGET_COLS.index(v) for v in vars_to_plot if v in TARGET_COLS]
    fig, axes = plt.subplots(len(indices), 2, figsize=(15, 3.2 * len(indices)))

    for row, c_idx in enumerate(indices):
        name = TARGET_COLS[c_idx]
        r_curve = real_ntc[:, c_idx]
        g_curves = gen_ntc[:, :, c_idx]
        g_mean = g_curves.mean(axis=0)
        g_std = g_curves.std(axis=0)

        # Colonna 1: Traiettorie temporali (Reale vs Ensemble Generato)
        ax_t = axes[row, 0]
        for k in range(n_ensemble):
            ax_t.plot(t_axis, g_curves[k], color="tab:blue", alpha=0.2, lw=1)
        ax_t.plot(t_axis, g_mean, color="tab:blue", ls="--", lw=2, label="Media Generata")
        ax_t.fill_between(t_axis, g_mean - g_std, g_mean + g_std, color="tab:blue", alpha=0.15, label="±1 std")
        ax_t.plot(t_axis, r_curve, color="crimson", lw=2.2, label="Reale (Ground Truth)")
        ax_t.set_title(f"{name} — Dinamica Temporale")
        ax_t.grid(True, alpha=0.3)
        if row == 0:
            ax_t.legend(loc="upper right", fontsize=8)

        # Colonna 2: Contenuto in frequenza (Spettro di Potenza / PSD)
        ax_f = axes[row, 1]
        freqs = np.fft.rfftfreq(len(r_curve), d=1.0)
        r_psd = np.abs(np.fft.rfft(r_curve - r_curve.mean())) ** 2
        g_psd = (np.abs(np.fft.rfft(g_curves - g_curves.mean(axis=1, keepdims=True), axis=1)) ** 2).mean(axis=0)
        ax_f.plot(freqs, r_psd, color="crimson", lw=2, label="PSD Reale")
        ax_f.plot(freqs, g_psd, color="tab:blue", ls="--", lw=2, label="PSD Media Generata")
        ax_f.set_title(f"{name} — Spettro di Frequenza (PSD)")
        ax_f.set_xlabel("Frequenza")
        ax_f.grid(True, alpha=0.3)
        if row == 0:
            ax_f.legend(loc="upper right", fontsize=8)

    plt.tight_layout()
    plt.show()

# Chiamata alla funzione di visualizzazione
plot_thesis_diagnostics(process, test_ds, scaler, sample_idx=10, n_ensemble=15)